# VNICT2026 GeoFormerDock — kiem tra + chuan bi du lieu + training tren Kaggle

Notebook nay lam 5 viec, theo thu tu (`docs/revision_plan_reviews.md`):

1. **Tier 1** — kiem tra 5 gia tri `--geo_ablation` moi them vao GeoFormerDock.
2. **Tier 2** — smoke test pipeline training that tren 2 mau co san trong repo.
3. **Tier 3** — chi lay 2 file `.types` can dung, tach validation split.
4. **Tier 4** — chi trich cau truc protein cho DUNG cac receptor can dung.
5. **Track A** — training that (geoformerdock truoc).

**⚠️ TRUOC KHI CHAY BAT KY CELL NAO: vao Settings (panel phai) → Accelerator
→ GPU T4 x2, VA Internet → On.** Bat GPU tu dau — doi Accelerator GIUA CHUNG
notebook se **XOA SACH session** (da gap that: mat ca repo lan `data/` da tai).
Bat tu dau thi khong bao gio can doi giua chung nua.

**Sau khi bat GPU tu dau va notebook nay da het can restart kernel giua chung**
(xem Muc 0 duoi), co the dung **Save Version → "Save & Run All (Commit)"** de
chay toan bo tu dau den cuoi tren server Kaggle — **khong phu thuoc tab trinh
duyet/may tinh con mo hay khong**. Van co gioi han thoi luong phien cua rieng
Kaggle (thuong ~9-12h cho GPU) — chay khong xong trong 1 lan thi phai chia
nho hoac chay tiep phien sau.

Neu bat ky Tier nao FAIL: dung lai, dan output vao chat cho Claude, DUNG chay tiep.

## 0. Cai dat truoc tien — TRUOC KHI CHAY BAT KY CELL NAO KHAC

Cai `numpy<2` + `molgrid` + `ignite` + `mlflow` **ngay tu dau, truoc khi bat ky
thu gi import `numpy`/`torch`** trong phien nay. Ly do: `numpy` la C-extension,
neu no da duoc import (du chi de kiem tra phien ban) roi moi `pip install` ban
khac, tien trinh Python dang chay se KHONG thay ban moi — bat buoc phai restart
kernel. Cai truoc tien nhu the nay thi khong bao gio can restart nua, va toan
bo notebook chay duoc lien mach tu dau den cuoi bang **Save & Run All
(Commit)** — thu duy nhat Kaggle giu chay that su o server, khong phu thuoc
ban dong tab/tat may.

In [ ]:
!pip install -q 'numpy<2' molgrid mlflow
!pip install -q --no-deps pytorch-ignite


In [ ]:
import subprocess, sys
# QUAN TRONG: khong "import numpy/torch" TRUC TIEP o day (Kaggle preload numpy
# rieng vao kernel) — kiem tra qua subprocess. Cung kiem tra torch co phai ban
# CUDA khong: da tung gap that ca "pip install molgrid pytorch-ignite mlflow"
# vo tinh keo theo mot ban torch CPU-only (torch.__version__ += '+cpu' thay vi
# '+cuXXX'), lam moi lenh GPU sau nay loi "CUDA driver version is insufficient"
# ngay tu dau — kiem tra som o day de bao loi trong vai giay thay vi ~9 phut.
r = subprocess.run([sys.executable, '-c', '''
import numpy, torch, molgrid
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("molgrid: import OK")
assert numpy.__version__.startswith("1."), f"numpy={numpy.__version__} van >=2"
assert torch.cuda.is_available(), (
    f"torch.__version__={torch.__version__} KHONG thay GPU (co the pip install da "
    "vo tinh keo torch ve ban CPU-only, hoac Settings Accelerator chua bat GPU). "
    "Kiem tra Settings panel phai, hoac bao Claude kem dong nay."
)
print("OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr, file=sys.stderr)
    raise RuntimeError('Kiem tra thu vien THAT BAI - dan cho Claude.')


## 1. Lay code moi nhat tu GitHub

Repo public: `https://github.com/ducnm-mimhus/VNICT2026_Docking_Paper.git` — khong can dang nhap/lien ket tai khoan gi, chi can
Internet: On o buoc tren.

In [ ]:
import os
if os.path.isdir('/kaggle/working/VNICT2026_Docking_Paper'):
    print('/kaggle/working/VNICT2026_Docking_Paper da ton tai — pull thay vi clone lai')
    !cd /kaggle/working/VNICT2026_Docking_Paper && git pull
else:
    !cd /kaggle/working && git clone https://github.com/ducnm-mimhus/VNICT2026_Docking_Paper.git


In [ ]:
%cd /kaggle/working/VNICT2026_Docking_Paper
!git log --oneline -1
print()
print('>>> Dan dong commit tren vao chat de Claude xac nhan ban dang chay ban moi nhat.')


## 2. Tier 1 — kiem tra `--geo_ablation` (KHONG can GPU, KHONG can `data/`)

Chi dung `torch` (Kaggle da cai san). Kiem tra:
1. Ca 5 gia tri `--geo_ablation` deu construct + forward pass duoc.
2. `geo_ablation="none"` cho DUNG 1.594.573 tham so (bang so da bao cao trong bai).
3. Checkpoint cu (neu co trong repo) load duoc vao model "none" moi, khong
   missing/unexpected keys.

**Neu cell duoi FAIL: dung lai, dan toan bo output cho Claude, DUNG chay Tier 2/3.**

In [ ]:
!python3 tools/verify_geo_ablation.py


## 3. Tier 2 — smoke test pipeline training that (chi chay neu Tier 1 PASS)

`molgrid`/`ignite`/`mlflow` da duoc cai o Muc 0 (dau notebook) — khong can cai
lai o day.

Dung `demo_inference/` co san trong repo (4 file `.gninatypes` + 1 file `.types`,
vai chuc KB) — KHONG tai `data/` 80GB. Chay 3 epoch tren 2 mau, mat vai giay.

In [ ]:
%cd /kaggle/working/VNICT2026_Docking_Paper
!python -u -m dockbench.training \
    demo_inference/types/demo.types \
    -d demo_inference \
    -m geoformerdock \
    --label_pos 0 --affinity_pos 1 \
    --batch_size 2 \
    -i 3 \
    --iteration_scheme small \
    --max_pseudo_atoms 12 \
    --seed 2026 \
    -o /tmp/tier2_smoketest
!echo '--- Neu khong co loi/traceback o tren la PASS (chi can chay het, khong can ket qua tot) ---'


## 4. Tier 3 — chi lay 2 file `.types` can dung (KHONG tai nguyen `paper_types.tar.gz` 6.5GB)

`paper_types.tar.gz` gop chung `.types` cua RAT NHIEU bo split khac nhau cua
CrossDocked2020, nhung du an chi can dung 2 file: `ref_uff_train0.types` va
`ref_uff_test0.types`. Cell duoi **stream truc tiep qua pipe** (`curl | tar`),
KHONG BAO GIO ghi file `.tar.gz` 6.5GB xuong dia — chi ghi ra dung 2 file
`.types` can (vai chuc MB). Da tung thu cach nay tren may lab (bang thong rat
cham, ~8-10 KB/s) va bi timeout; Kaggle thuong co bang thong quoc te tot hon
nhieu nen cell nay CO THE nhanh hon dang ke — neu van cham/timeout, bao lai
Claude ngay, dung thu tiep.

In [ ]:
!mkdir -p data/types
!timeout 1800 curl -sL 'https://bits.csb.pitt.edu/files/crossdock2020/v1.0/paper_types.tar.gz' \
    | tar -xz -C data types/ref_uff_train0.types types/ref_uff_test0.types
!echo '--- ket qua ---'
!ls -la data/types/ 2>&1
!wc -l data/types/ref_uff_train0.types data/types/ref_uff_test0.types 2>&1


**Kiem tra truoc khi chay tiep:** cell tren phai in ra 2 file voi so dong
khop ky vong: `ref_uff_train0.types` = 62.335 dong, `ref_uff_test0.types` =
4.618 dong. Neu bao 'No such file or directory' hoac so dong = 0, DUNG lai —
bao Claude ngay, dung doan tiep vi cac buoc sau deu can 2 file nay.

In [ ]:
!python3 tools/make_val_split.py \
    --train data/types/ref_uff_train0.types \
    --out_train data/types/ref_uff_train0_split.types \
    --out_val data/types/ref_uff_val0.types \
    --val_frac 0.15 --seed 2026


**Kiem tra A0b (doc trong `docs/revision_plan_reviews.md`):** trong output cua
cell tren phai co dong `Giao receptor train/val (phai = 0)          : 0`.
Neu con so cuoi khac 0, DUNG lai va bao Claude — co loi ro ri du lieu.

In [ ]:
# Dem so mau y_aff > 0 trong tap val moi tach (A0b: can >= 500 de C-index
# tren val du on dinh de chon checkpoint).
def count_pos(path):
    n_lines = n_pos = 0
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            n_lines += 1
            if float(line.split()[1]) > 0:
                n_pos += 1
    return n_lines, n_pos

for split in ['train0_split', 'val0']:
    n, npos = count_pos(f'data/types/ref_uff_{split}.types')
    print(f'{split:14s}: {n:7d} dong, {npos:5d} mau y_aff>0 ({100*npos/n:.1f}%)')


## 5. Tier 4 — chi trich cau truc protein (PDBbind2016) cho DUNG cac receptor can dung

`PDBbind2016.tar.gz` (4.2GB nen) chua cau truc cua **toan bo** receptor trong
CrossDocked2020, nhung `ref_uff_train0`/`test0` chi tham chieu toi mot tap con.
Cell duoi:
1. Doc 2 file `.types` da co, lay danh sach **receptor rieng biet** duoc tham chieu.
2. Ghi danh sach do thanh file (moi dong 2 duong dan ung vien — vi CHUA BIET
   chac chan archive dung tien to `PDBbind2016/<ma>/` hay chi `<ma>/` — dua ca
   hai vao, `tar` se tu bo qua duong dan nao khong khop, khong loi).
3. Stream-extract CHI cac thu muc receptor do tu `PDBbind2016.tar.gz`, khong
   bao gio ghi file .tar.gz 4.2GB xuong dia.

In [ ]:
receptors = set()
for fname in ['data/types/ref_uff_train0.types', 'data/types/ref_uff_test0.types']:
    with open(fname) as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.split()
            # cot 3 (0-indexed) la receptor_path, dang '<ma>/<file>.gninatypes'
            receptor_path = parts[3]
            receptors.add(receptor_path.split('/')[0])

print(f'So receptor rieng biet can dung: {len(receptors)}')

with open('receptor_members.txt', 'w') as f:
    for r in sorted(receptors):
        f.write(f'PDBbind2016/{r}\n')  # ung vien 1: co tien to PDBbind2016/
        f.write(f'{r}\n')              # ung vien 2: khong co tien to

print(f'Da ghi receptor_members.txt ({len(receptors) * 2} dong, 2 ung vien/receptor)')


In [ ]:
# Chay curl|tar o NEN, tu kiem tra tien do moi 20s, TU DUNG khi:
#   (a) da co >=98% receptor can dung, HOAC
#   (b) 80s lien tuc khong tang them (nghia la da doc qua het cac muc can,
#       phan con lai cua archive khong con gi lien quan), HOAC
#   (c) qua 60 phut (gioi han an toan, tranh treo vo han).
# Giai quyet dung van de gap o Tier 3: tar doc tu pipe khong biet khi nao
# da du, se cu doc het luong neu khong ai chu dong dung no lai.
import os, signal, subprocess, time

os.makedirs('data', exist_ok=True)
# set -o pipefail: de returncode phan anh dung loi cua curl (vd 404) chu
# khong chi loi cua tar (mac dinh shell chi lay returncode lenh CUOI trong pipe).
cmd = "set -o pipefail; curl -sL 'https://bits.csb.pitt.edu/files/crossdock2020/PDBbind2016.tar.gz' | tar -xz -C data -T receptor_members.txt"
proc = subprocess.Popen(cmd, shell=True, executable='/bin/bash', preexec_fn=os.setsid)

def coverage():
    found = sum(
        1 for r in receptors
        if os.path.isdir(f'data/PDBbind2016/{r}') or os.path.isdir(f'data/{r}')
    )
    return found, len(receptors)

last_found, stable_checks, start = -1, 0, time.time()
MAX_WAIT_S = 3600

while True:
    time.sleep(20)
    found, total = coverage()
    elapsed = time.time() - start
    print(f'[{elapsed:5.0f}s] receptor da co du lieu: {found}/{total} ({100*found/total:.1f}%)')

    if proc.poll() is not None:
        rc = proc.returncode
        status = 'THANH CONG (doc het archive)' if rc == 0 else f'LOI (returncode={rc}) — kiem tra lai URL/mang'
        print(f'curl|tar da tu ket thuc: {status}.')
        break

    stable_checks = stable_checks + 1 if found == last_found else 0
    last_found = found

    if found / total >= 0.98:
        print('>>> Da dat >=98% receptor — dung tien trinh, bo qua phan con lai cua archive.')
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        break
    if stable_checks >= 4:
        print(f'>>> Khong tang them sau {stable_checks * 20}s — co le da doc qua het cac muc '
              f'lien quan. Dung tien trinh (con {total - found} receptor CO THE khong co trong archive).')
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        break
    if elapsed >= MAX_WAIT_S:
        print('>>> Qua 60 phut — dung tien trinh. Bao Claude ket qua nay.')
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        break

time.sleep(2)
print()
print('--- ket qua ---')
!du -sh data/ 2>&1
!df -h /kaggle/working 2>&1 | tail -1


**Kiem tra doc lap — KHONG doan cau truc thu muc, kiem tra thang tren danh
sach receptor da biet** (2 ung vien duong dan co the tao ra do sau thu muc
khac nhau, nen dem thu muc theo do sau se sai; kiem tra nay dung cho ca 2
truong hop):

In [ ]:
import os
found_prefixed = found_flat = missing = 0
missing_examples = []
for r in receptors:
    if os.path.isdir(f'data/PDBbind2016/{r}'):
        found_prefixed += 1
    elif os.path.isdir(f'data/{r}'):
        found_flat += 1
    else:
        missing += 1
        if len(missing_examples) < 5:
            missing_examples.append(r)

total = len(receptors)
print(f'Tong receptor can: {total}')
print(f'  Tim thay dang data/PDBbind2016/<ma>/  : {found_prefixed}')
print(f'  Tim thay dang data/<ma>/ (khong tien to): {found_flat}')
print(f'  KHONG tim thay o ca 2 dang             : {missing} '
      f'({100*missing/total:.1f}%)')
if missing_examples:
    print(f'  Vi du receptor bi thieu: {missing_examples}')
if missing / total > 0.05:
    print()
    print('CANH BAO: thieu hon 5% receptor — DUNG lai, bao Claude ngay kem '
          'output nay va output cell truoc.')
else:
    print()
    print('OK — da co du du lieu cho gan het receptor can dung.')


**Loc theo TUNG FILE cu the, khong chi theo thu muc** — da xac nhan that
(chay Track A that): `molgrid` **crash cung** khi gap dong `.types` tro toi
file khong doc duoc (`ValueError: Could not read ...`), khong bo qua em.
Kiem tra chi theo `os.path.isdir()` (thu muc receptor co ton tai) LA CHUA
DU: co che tu dung o Tier 4 (`killpg`) co the cat ngang dung luc `tar` dang
ghi do mot file, de lai thu muc "co ve day du" nhung mot vai file ben trong
bi thieu/dang do. Cell duoi kiem tra THANG tung file (ca receptor lan
ligand) co doc duoc that khong, khong suy doan tu thu muc cha.

In [ ]:
import os

def validate_and_filter(path):
    with open(path) as f:
        lines = [ln for ln in f if ln.strip()]
    kept, removed = [], 0
    for ln in lines:
        parts = ln.split()
        receptor_path, ligand_path = parts[3], parts[4]
        if os.path.isfile(f'data/{receptor_path}') and os.path.isfile(f'data/{ligand_path}'):
            kept.append(ln)
        else:
            removed += 1
    with open(path, 'w') as f:
        f.writelines(kept)
    print(f'{path}: {len(lines)} -> {len(kept)} dong '
          f'(loai {removed} dong co file thuc su thieu/hong)')

for p in ['data/types/ref_uff_train0_split.types',
          'data/types/ref_uff_val0.types',
          'data/types/ref_uff_test0.types']:
    validate_and_filter(p)


## 6. Tom tat — dan phan nay vao chat cho Claude

Chay cell duoi roi copy toan bo output gui lai, kem ket qua Tier 1-4 o tren.

In [ ]:
import subprocess, sys
# QUAN TRONG: khong "import numpy/torch" TRUC TIEP o day (Kaggle preload numpy
# rieng vao kernel) — kiem tra qua subprocess. Cung kiem tra torch co phai ban
# CUDA khong: da tung gap that ca "pip install molgrid pytorch-ignite mlflow"
# vo tinh keo theo mot ban torch CPU-only (torch.__version__ += '+cpu' thay vi
# '+cuXXX'), lam moi lenh GPU sau nay loi "CUDA driver version is insufficient"
# ngay tu dau — kiem tra som o day de bao loi trong vai giay thay vi ~9 phut.
r = subprocess.run([sys.executable, '-c', '''
import numpy, torch, molgrid
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("molgrid: import OK")
assert numpy.__version__.startswith("1."), f"numpy={numpy.__version__} van >=2"
assert torch.cuda.is_available(), (
    f"torch.__version__={torch.__version__} KHONG thay GPU (co the pip install da "
    "vo tinh keo torch ve ban CPU-only, hoac Settings Accelerator chua bat GPU). "
    "Kiem tra Settings panel phai, hoac bao Claude kem dong nay."
)
print("OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr, file=sys.stderr)
    raise RuntimeError('Kiem tra thu vien THAT BAI - dan cho Claude.')


## 7. Track A — training that (BAT DAU voi geoformerdock, seed 2026)

GPU da duoc bat tu dau (Muc 0 tren cung) — khong can lam gi them o day, chay
thang cell duoi.

Sieu tham so duoi day sao chep NGUYEN VAN tu `train_one()` trong
`scripts/run_overnight_valsplit.sh` (chi bo phan sourcing conda va
wait_for_gpu — Kaggle da cap san dung 1 GPU cho phien nay, khong can doi).
Chay `geoformerdock` truoc — quan trong nhat cho luan diem chinh cua bai.

In [ ]:
%%bash
TRAIN_FILE=data/types/ref_uff_train0_split.types
VAL_FILE=data/types/ref_uff_val0.types
TEST_FILE=data/types/ref_uff_test0.types
SEED=2026
MODEL=geoformerdock
OUTDIR=results/models/${MODEL}_valsplit_s${SEED}
LOGFILE=results/logs/${MODEL}_valsplit_s${SEED}.log
mkdir -p results/logs

for bs in 256 128 64; do
    echo "=== ${MODEL}: thu batch_size=${bs} ($(date)) ==="
    rm -rf "${OUTDIR}"
    START=$(date +%s)
    if python -u -m dockbench.training \
        "${TRAIN_FILE}" \
        --testfile "${TEST_FILE}" \
        --valfile "${VAL_FILE}" \
        -d data \
        -m "${MODEL}" \
        --label_pos 0 --affinity_pos 1 \
        --base_lr 0.001 --weight_decay 0.01 \
        --batch_size "${bs}" \
        --random_translation 1.0 --clip_gradients 5.0 \
        -i 100 \
        --iteration_scheme small \
        --lr_dynamic --warmup_epochs 2 \
        --test_every 2 --checkpoint_every 100 \
        --no_roc_auc \
        --scale_affinity_loss 1.0 --delta_affinity_loss 1.0 \
        --scale_ranking 0.05 --ranking_temperature 1.0 --ranking_num_pairs 128 \
        --hard_neg_fraction 0.3 \
        --rank_warmup_epochs 10 --rank_rampup_epochs 15 \
        --scale_pose_coupling 0.00 --lambda_pose 1.2 \
        --pose_warmup_epochs 0 --pose_only_epochs 4 \
        --pose_loss_type focal --pose_focal_gamma 2.0 --pose_focal_alpha 0.75 \
        --pose_class_normalize --pose_balance_batch --pose_balance_target_per_class 32 \
        --pose_prior_logit_scale 0.25 --disable_pose_prior_init \
        --pose_loss_scale 0.5 --pose_total_weight 0.85 --aff_total_weight 0.15 \
        --metric_ema_alpha 0.3 \
        --early_stop_metric composite_cidx_balacc --early_stop_composite_w_cidx 0.5 \
        --early_stop_patience 25 --early_stop_min_delta 0.0001 \
        --scale_dist_constraint 0.02 --scale_anchor_loss 0.01 \
        --normalize_targets --seed "${SEED}" \
        --max_pseudo_atoms 12 --use_amp \
        -g cuda:0 \
        -o "${OUTDIR}" \
        2>&1 | tee "${LOGFILE}"
    then
        if [ -f "${OUTDIR}/summary.json" ]; then
            ELAPSED=$(( $(date +%s) - START ))
            echo "=== ${MODEL}: THANH CONG voi batch_size=${bs}, mat ${ELAPSED}s (~$(( ELAPSED / 60 )) phut) ==="
            exit 0
        fi
    fi
    echo "=== ${MODEL}: THAT BAI o batch_size=${bs}, thu nho hon ==="
done
echo "=== ${MODEL}: THAT BAI CA 3 MUC batch_size ===" >&2
exit 1


**Kiem tra ket qua:** cell tren phai ket thuc voi dong `THANH CONG voi
batch_size=...`, va co so phut thuc te da chay. Dan dong do cho Claude — se
dung de uoc tinh xem 3 mo hinh con lai (`gnina_dense`, `gnina_default2018`,
`pafnucy`) co kip trong phien hien tai khong.

**Kiem tra nhanh `summary.json`:**

In [ ]:
import json
d = json.load(open('results/models/geoformerdock_valsplit_s2026/summary.json'))
print({k: d[k] for k in ('selection_split', 'best_epoch', 'best_val_score', 'pose_threshold', 'final_bal_acc', 'final_pr_auc', 'final_c_index')})
